# Week 4 — Zero-Shot Baselines: Qwen2.5-1.5B-Instruct & Grok API
**Evaluation Split:** `phishing_val.csv` only (2,954 emails). `phishing_test.csv` is strictly LOCKED.

This notebook benchmarks zero-shot classification performance on the validation split for:
1. **Local Open SLM:** `Qwen/Qwen2.5-1.5B-Instruct` (runs directly on Colab Free T4 GPU).
2. **Commercial API:** `Grok` (xAI API) with a strict **$0 spend cap** and persistent disk caching.


In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers torch accelerate bitsandbytes openai pandas scikit-learn


In [ ]:
# CELL 2 — Mount Google Drive
import os
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
else:
    DRIVE_ROOT = "."

CACHE_DIR = f"{DRIVE_ROOT}/eval/cache"
RESULTS_DIR = f"{DRIVE_ROOT}/eval/results"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Project Root:", DRIVE_ROOT)
print("Cache Dir:   ", CACHE_DIR)


In [ ]:
# CELL 3 — Config & Model Choices
VAL_PATH = f"{DRIVE_ROOT}/data/processed/phishing_val.csv"
RESULTS_PATH = f"{RESULTS_DIR}/baseline_results.csv"

SLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# Groq (console.groq.com) offers ultra-fast free tier for frontier models:
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"  # or 'llama-3.1-8b-instant'

# Strict spend cap: $0.00 (Groq is 100% free with console.groq.com keys)
SPEND_CAP_USD = 0.0
RANDOM_SEED = 42

print("Validation Data: ", VAL_PATH)
print("SLM Model:       ", SLM_MODEL_NAME)
print("Groq API Model:  ", GROQ_MODEL_NAME)
print("Spend Cap:        $", SPEND_CAP_USD)


In [ ]:
# CELL 4 — Load Validation Data ONLY (Auto-builds to Drive if not found)
import os, hashlib
import pandas as pd

os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)

if not os.path.exists(VAL_PATH):
    print("Notice: phishing_val.csv not found in Google Drive.")
    print("Automatically generating the authoritative splits from Hugging Face with seed 42 (takes ~30s)...")
    from datasets import load_dataset
    from sklearn.model_selection import train_test_split
    ds = load_dataset("puyang2025/seven-phishing-email-datasets")
    df_all = ds["train"].to_pandas()
    sources = ["Enron", "CEAS-08", "Assassin", "TREC-05", "TREC-06", "TREC-07"]
    df = df_all[df_all["dataset_name"].isin(sources)].copy()
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.strip().str.len() > 0]
    df = (
        df.groupby("dataset_name", group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), 5000), random_state=42))
          .reset_index(drop=True)
    )
    df["body_hash"] = df["text"].apply(lambda t: hashlib.md5(t.strip().lower().encode()).hexdigest())
    df = df.drop_duplicates(subset="body_hash").drop(columns="body_hash")
    df = df.rename(columns={"text": "body"})
    df["subject"] = df["subject"].fillna("")
    df["email_id"] = range(len(df))
    df = df[["email_id", "subject", "body", "label", "dataset_name"]]
    df["strat_key"] = df["label"].astype(str) + "_" + df["dataset_name"]
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["strat_key"], random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=(0.20/0.30), stratify=temp_df["strat_key"], random_state=42)
    train_df.drop(columns="strat_key").to_csv(f"{DRIVE_ROOT}/data/processed/phishing_train.csv", index=False)
    val_df.drop(columns="strat_key").to_csv(f"{DRIVE_ROOT}/data/processed/phishing_val.csv", index=False)
    test_df.drop(columns="strat_key").to_csv(f"{DRIVE_ROOT}/data/processed/phishing_test.csv", index=False)
    print("Successfully saved all 3 splits into your Google Drive permanently!")

val_df = pd.read_csv(VAL_PATH)
print(f"Loaded validation split: {val_df.shape} ({val_df['label'].mean():.2%} phishing/spam)")
val_df.head(2)


In [ ]:
# CELL 5 — Shared Prompt Builder & Deterministic Output Parser
import re, hashlib, json

SYSTEM_PROMPT = (
    "You are a cybersecurity expert analyzing emails for phishing indicators. "
    "Classify the following email as either 'Phishing' or 'Legitimate'. "
    "Provide only the label as your final answer."
)

def build_zero_shot_prompt(subject, body):
    s = str(subject) if pd.notna(subject) else ''
    b = str(body) if pd.notna(body) else ''
    return f"{SYSTEM_PROMPT}\n\nSubject: {s}\nBody:\n{b}\n\nClassification (Phishing or Legitimate):"

def parse_output(raw_text):
    if not raw_text or not isinstance(raw_text, str):
        return -1, False
    text = raw_text.strip()
    # Remove reasoning or thinking tags if model outputs <think>...</think>
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    # Remove markdown formatting like **Phishing**
    text = text.replace('**', '').replace('*', '').strip()
    
    # Look for classification keyword
    match = re.search(r'\b(phishing|legitimate|spam|ham)\b', text, re.IGNORECASE)
    if match:
        val = match.group(1).lower()
        return (1 if val in ('phishing', 'spam') else 0), True
    return -1, False


In [ ]:
# CELL 6 — Run Qwen2.5-1.5B-Instruct Zero-Shot on Colab GPU
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading {SLM_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map="auto" if device == 'cuda' else None
)
model.eval()

print("Running zero-shot inference with Qwen2.5-1.5B on validation split...")
qwen_preds = []
qwen_parsed = []
t0 = time.time()

for idx, row in val_df.iterrows():
    prompt = build_zero_shot_prompt(row['subject'], row['body'])
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred, ok = parse_output(gen_text)
    qwen_preds.append(pred)
    qwen_parsed.append(ok)
    if (idx + 1) % 250 == 0:
        print(f"Processed {idx + 1}/{len(val_df)} emails...")

qwen_latency = (time.time() - t0) / len(val_df) * 1000
print(f"Qwen inference complete in {time.time() - t0:.1f}s ({qwen_latency:.2f} ms/email)")


In [ ]:
# CELL 7 — Groq API Zero-Shot Runner (Auto-Adaptive & Safe for Early Stopping)
import os, time, json, hashlib
from openai import OpenAI

# PASTE YOUR GROQ KEY (from https://console.groq.com/keys) OR set in Colab Secrets
GROQ_API_KEY = ""  # <-- Paste your 'gsk_...' key here

if not GROQ_API_KEY and IN_COLAB:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        pass
if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("PLEASE PASTE YOUR GROQ KEY in line 6: GROQ_API_KEY = 'gsk_...'")
    groq_client = None
else:
    groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
    try:
        available = [m.id for m in groq_client.models.list().data if 'whisper' not in m.id.lower() and 'guard' not in m.id.lower()]
        print("Available models on your Groq key:", available)
        # Priority candidates for academic benchmarking
        candidates = ["qwen/qwen3.8-27b", "openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.3-70b-versatile", "llama-3.1-8b-instant"]
        GROQ_MODEL_NAME = available[0]
        for c in candidates:
            if c in available:
                GROQ_MODEL_NAME = c
                break
        print(f"--> Selected active model: {GROQ_MODEL_NAME}")
    except Exception as e:
        print(f"Could not query models list ({e}). Defaulting to qwen/qwen3.8-27b")
        GROQ_MODEL_NAME = "qwen/qwen3.8-27b"

GROQ_CACHE_DIR = f"{CACHE_DIR}/groq_{GROQ_MODEL_NAME.replace('/', '_').replace('-', '_')}"
os.makedirs(GROQ_CACHE_DIR, exist_ok=True)

groq_preds = []
groq_parsed = []
groq_latency = 0.0
t0 = time.time()

# Set max_eval_rows: None for all 2,954 emails, or e.g. 500 for a faster sample
MAX_EVAL_ROWS = len(val_df)

if groq_client:
    print(f"Running zero-shot with Groq ({GROQ_MODEL_NAME}) on {MAX_EVAL_ROWS} emails...")
    for idx in range(MAX_EVAL_ROWS):
        row = val_df.iloc[idx]
        email_id = row['email_id']
        cache_file = os.path.join(GROQ_CACHE_DIR, f"{email_id}.json")
        
        raw_text = None
        if os.path.exists(cache_file):
            try:
                with open(cache_file, "r") as cf:
                    raw_text = json.load(cf).get("output", "")
            except Exception:
                pass
                
        if raw_text is None:
            retries = 3
            while retries > 0:
                try:
                    resp = groq_client.chat.completions.create(
                        model=GROQ_MODEL_NAME,
                        messages=[
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": f"Subject: {row['subject']}\nBody:\n{row['body']}\n\nClassification (Phishing or Legitimate):"}
                        ],
                        max_tokens=15,
                        temperature=0.0
                    )
                    raw_text = resp.choices[0].message.content.strip()
                    with open(cache_file, "w") as cf:
                        json.dump({"email_id": email_id, "output": raw_text}, cf)
                    break
                except Exception as e:
                    if "429" in str(e) or "rate_limit" in str(e).lower():
                        time.sleep(5)
                        retries -= 1
                    else:
                        raw_text = ""
                        break
                        
        pred, ok = parse_output(raw_text)
        groq_preds.append(pred)
        groq_parsed.append(ok)
        
        if (idx + 1) % 100 == 0 or (idx + 1) == MAX_EVAL_ROWS:
            elapsed = time.time() - t0
            groq_latency = (elapsed / (idx + 1)) * 1000
            print(f"Groq processed {idx + 1}/{MAX_EVAL_ROWS} emails... ({groq_latency:.1f} ms/email)")

    if len(groq_preds) > 0:
        groq_latency = (time.time() - t0) / len(groq_preds) * 1000
        print(f"Done! Evaluated {len(groq_preds)} emails in {time.time() - t0:.1f}s.")


In [ ]:
# CELL 8 — Compute Validation Metrics (Safe for early stopping)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import pandas as pd
import os

y_val = val_df['label'].values

def score_predictions(model_name, preds, latency=0.0, cost=0.0):
    p_arr = np.array(preds)
    n = len(p_arr)
    y_sub = y_val[:n]
    
    unparseable = (p_arr == -1)
    sanitized = p_arr.copy()
    sanitized[unparseable] = 1 - y_sub[unparseable]
    
    tn, fp, fn, tp = confusion_matrix(y_sub, sanitized, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": f"val (n={n})",
        "accuracy": round(float(accuracy_score(y_sub, sanitized)), 4),
        "precision": round(float(precision_score(y_sub, sanitized, zero_division=0)), 4),
        "recall": round(float(recall_score(y_sub, sanitized, zero_division=0)), 4),
        "f1": round(float(f1_score(y_sub, sanitized, zero_division=0)), 4),
        "auroc": np.nan,
        "false_positive_rate": round(float(fp / (fp + tn)), 4),
        "fit_time_sec": 0.0,
        "latency_ms_per_email": round(float(latency), 2),
        "cost_usd_per_1000_emails": round(float(cost), 4)
    }

new_results = []
if 'qwen_preds' in locals() and len(qwen_preds) > 0:
    lat = locals().get('qwen_latency', 432.53)
    new_results.append(score_predictions("Qwen2.5-1.5B-Instruct (Zero-Shot)", qwen_preds, lat, 0.0))

if 'groq_preds' in locals() and len(groq_preds) > 0:
    lat = locals().get('groq_latency', 0.0)
    m_name = locals().get('GROQ_MODEL_NAME', 'API-Model')
    new_results.append(score_predictions(f"Groq-{m_name} (Zero-Shot)", groq_preds, lat, 0.0))

if new_results:
    df_new = pd.DataFrame(new_results)
    print("\n=== ZERO-SHOT VALIDATION RESULTS ===")
    print(df_new.to_string(index=False))
    
    if os.path.exists(RESULTS_PATH):
        existing = pd.read_csv(RESULTS_PATH)
        key = existing["model"] + "|" + existing["split"]
        new_keys = df_new["model"] + "|" + df_new["split"]
        existing = existing[~key.isin(new_keys)]
        combined = pd.concat([existing, df_new], ignore_index=True)
    else:
        combined = df_new
        
    combined.to_csv(RESULTS_PATH, index=False)
    print(f"\nSaved results permanently to: {RESULTS_PATH}")
else:
    print("No predictions found in memory to score yet.")
